In [1]:
import pandas as pd

df = pd.read_csv('FactInventoryAdjustments.csv', encoding='latin1')

print(df.shape)
print(df.head())
df.dtypes


(2120, 24)
  Facility ID Facility Name Adjustment Type Reason Code ID  \
0       D7502         D7502      PACK_SHORT           EN27   
1       D7502         D7502      PACK_SHORT           EN27   
2       D7502         D7502     MODIFY_ILPN           FN28   
3       D7502         D7502     MODIFY_ILPN           FN28   
4       D7502         D7502     CYCLE_COUNT           AD01   

  Adjustment Reason Description  ASN Status Purchase Order ID    Item ID  \
0             Put to Store Void      3000.0       S6100675635  100530865   
1             Put to Store Void      3000.0       S6100637841  100000066   
2      FN28 - Void to Reinstate      3000.0       S4100402307  100123848   
3      FN28 - Void to Reinstate      3000.0       S4100418198  100130665   
4                   Cycle Count         NaN               NaN  100113976   

  Order Type                             Item Description  ...  Unit Qty  \
0    Stocked              100530865 Quavers MIXCHE 20X16G  ...       -14   
1    St

Facility ID                       object
Facility Name                     object
Adjustment Type                   object
Reason Code ID                    object
Adjustment Reason Description     object
ASN Status                       float64
Purchase Order ID                 object
Item ID                            int64
Order Type                        object
Item Description                  object
6 Digit Vendor ID                float64
Vendor Name                       object
Location ID                       object
ILPN ID                          float64
Unit Qty                           int64
SubPack Qty                      float64
SubPack Qty GROSS                float64
WAC Item Cost                    float64
Total WAC                        float64
Total WAC Gross                  float64
Total PO Cost                    float64
PO Cost Gross                    float64
Created By                        object
Facility - Created Timestamp      object
dtype: object

In [2]:
#Item ID/ Location ID will be the key to reference other tables
#Order Type, ILPN ID, Adjustment Reason Description will be used for filtering and grouping

wanted = ['Adjustment Type', 'Adjustment Reason Description', 'Item ID', 'Order Type',
          'Location ID', 'ILPN ID', 'Unit Qty', 'SubPack Qty', 'Created By', 'Facility - Created Timestamp']

df_clean = df[wanted].copy()

print(df_clean.shape)
df_clean.head()


(2120, 10)


,Adjustment Type,Adjustment Reason Description,Item ID,Order Type,Location ID,ILPN ID,Unit Qty,SubPack Qty,Created By,Facility - Created Timestamp
0,PACK_SHORT,Put to Store Void,100530865,Stocked,SL001,37268479.0,-14,-2.0,k0g08qv,20/12/2025 11:55
1,PACK_SHORT,Put to Store Void,100000066,Stocked,SL001,38152148.0,-4,-2.0,k0g08qv,20/12/2025 12:34
2,MODIFY_ILPN,FN28 - Void to Reinstate,100123848,Stocked,SL003,149212.0,-800,-80.0,j0g01i9,20/12/2025 16:41
3,MODIFY_ILPN,FN28 - Void to Reinstate,100130665,Stocked,SL004,38670115.0,-2240,-160.0,j0g01i9,20/12/2025 16:47
4,CYCLE_COUNT,Cycle Count,100113976,Stocked,G126026,NaN,42,3.0,a0g00c8,20/12/2025 20:29


In [3]:
df_clean = df_clean[
    (df_clean['Adjustment Reason Description'] == 'Cycle Count') &
    (df_clean['Order Type'] == 'Stocked') &
    (df_clean['ILPN ID'].isnull()) &
    (df_clean['Location ID'].notnull())
]

print(f"Rows remaining: {df_clean.shape[0]}")
df_clean.head()


Rows remaining: 1572


,Adjustment Type,Adjustment Reason Description,Item ID,Order Type,Location ID,ILPN ID,Unit Qty,SubPack Qty,Created By,Facility - Created Timestamp
4,CYCLE_COUNT,Cycle Count,100113976,Stocked,G126026,NaN,42,3.0,a0g00c8,20/12/2025 20:29
5,CYCLE_COUNT,Cycle Count,100139182,Stocked,F252052,NaN,-21,-3.0,a0g00c8,20/12/2025 20:30
6,CYCLE_COUNT,Cycle Count,100121295,Stocked,F105005,NaN,12,4.0,a0g00c8,20/12/2025 20:38
7,CYCLE_COUNT,Cycle Count,100115208,Stocked,E627027,NaN,90,3.0,a0g00c8,20/12/2025 20:47
8,CYCLE_COUNT,Cycle Count,100121448,Stocked,E926026,NaN,72,18.0,a0g00c8,20/12/2025 20:49


In [4]:
#Reset Index after filtering
df_clean.reset_index(drop=True, inplace=True)

In [5]:
# Now we can remove ILPN ID, Order Type, Adjustment Reason Description, Adjustment Type since they are no longer needed for analysis

df_clean.drop(columns=['ILPN ID', 'Order Type', 'Adjustment Reason Description', 'Adjustment Type'], inplace=True)
df_clean.head()

,Item ID,Location ID,Unit Qty,SubPack Qty,Created By,Facility - Created Timestamp
0,100113976,G126026,42,3.0,a0g00c8,20/12/2025 20:29
1,100139182,F252052,-21,-3.0,a0g00c8,20/12/2025 20:30
2,100121295,F105005,12,4.0,a0g00c8,20/12/2025 20:38
3,100115208,E627027,90,3.0,a0g00c8,20/12/2025 20:47
4,100121448,E926026,72,18.0,a0g00c8,20/12/2025 20:49


In [6]:
# Convert SubPack Qty and Item ID
df_clean['SubPack Qty'] = df_clean['SubPack Qty'].astype(int)
df_clean['Item ID'] = df_clean['Item ID'].astype(str)


In [7]:
# Rename and standardise column names first
df_clean.rename(columns={'Facility - Created Timestamp': 'created_timestamp'}, inplace=True)
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')

In [8]:
#convert timestamp
df_clean['created_timestamp'] = pd.to_datetime(df_clean['created_timestamp'], dayfirst=True)

print(df_clean.dtypes)
df_clean.head()

item_id                      object
location_id                  object
unit_qty                      int64
subpack_qty                   int64
created_by                   object
created_timestamp    datetime64[ns]
dtype: object


,item_id,location_id,unit_qty,subpack_qty,created_by,created_timestamp
0,100113976,G126026,42,3,a0g00c8,2025-12-20 20:29:00
1,100139182,F252052,-21,-3,a0g00c8,2025-12-20 20:30:00
2,100121295,F105005,12,4,a0g00c8,2025-12-20 20:38:00
3,100115208,E627027,90,3,a0g00c8,2025-12-20 20:47:00
4,100121448,E926026,72,18,a0g00c8,2025-12-20 20:49:00


In [9]:
print(f"Duplicate rows: {df_clean.duplicated().sum()}")


Duplicate rows: 0


In [10]:
df_clean.to_csv('FactInventoryAdjustments_cleaned.csv', index=False)